In [ ]:
# install required libraries
!pip install pandas scikit-learn transformers torch

In [38]:
# import required libraries for the program
import pandas as pd
import re

from transformers import pipeline
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

In [39]:
# load the dataset
from google.colab import drive
drive.mount('/content/gdrive')
dataset = pd.read_csv('/content/gdrive/My Drive/IMDB(Horror-1911-2018).csv')

Drive already mounted at /content/gdrive; to attempt to forcibly remount, call drive.mount("/content/gdrive", force_remount=True).


In [40]:
# display dataset information
print("Dataset Shape:")
print(dataset.shape)

print("\nDataset Columns:")
print(dataset.columns.tolist())

print("\nFirst 5 Rows:")
print(dataset.head())

Dataset Shape:
(19594, 11)

Dataset Columns:
['Unnamed: 0', 'Id', 'movie', 'genre', 'des', 'crew', 'year', 'imdb', 'votes', 'bag', 'key']

First 5 Rows:
   Unnamed: 0         Id                            movie  \
0       16014  tt1083844                            Chill   
1        1059  tt0277412                      Unspeakable   
2        3621  tt0114070                      The Outpost   
3        7114  tt0098090         The Phantom of the Opera   
4       11245  tt0071276  Captain Kronos - Vampire Hunter   

                        genre  \
0            Horror, Thriller   
1            Horror, Thriller   
2                      Horror   
3        Drama, Horror, Music   
4  Adventure, Horror, Mystery   

                                                 des  \
0  In this classic retro horror thriller, Sam, an...   
1             A woman battles an unspeakable terror.   
2  Government scientists attempt to reanimate a c...   
3  A young soprano becomes the obsession of a hor...   
4

In [41]:
# keep only needed columns
dataset = dataset[["des", "imdb"]].copy()

# rename columns
dataset = dataset.rename(columns={
    "des": "text",
    "imdb": "rating"
})

# remove empty rows
dataset = dataset.dropna(subset=["text", "rating"])

# convert rating to numeric
dataset["rating"] = pd.to_numeric(dataset["rating"], errors="coerce")
dataset = dataset.dropna(subset=["rating"])

# create clearer sentiment labels
dataset = dataset[
    (dataset["rating"] >= 7.5) |
    (dataset["rating"] <= 4.5)
].copy()

dataset["true_label"] = dataset["rating"].apply(
    lambda x: "POSITIVE" if x >= 7.5 else "NEGATIVE"
)

# clean and shorten text
dataset["text"] = dataset["text"].astype(str)
dataset = dataset[dataset["text"].str.len() > 20]
dataset["text"] = dataset["text"].str[:250]

# take balanced samples
positive = dataset[dataset["true_label"] == "POSITIVE"].sample(n=200, random_state=42)
negative = dataset[dataset["true_label"] == "NEGATIVE"].sample(n=200, random_state=42)

# combine and shuffle dataset
dataset = pd.concat([positive, negative])
dataset = dataset[["text", "true_label"]]
dataset = dataset.sample(frac=1, random_state=42).reset_index(drop=True)

# display final dataset
print(dataset.head())
print("\nClass Distribution:")
print(dataset["true_label"].value_counts())

                                                text true_label
0  It's death-by-fear (aka scared-to-death) in th...   NEGATIVE
1  As an Indie Horror flick director gathers his ...   NEGATIVE
2  A singing notebook tells 3 puppets to be creat...   POSITIVE
3  There are thin places between this world and t...   NEGATIVE
4  In a post-apocalyptic world, a family is force...   POSITIVE

Class Distribution:
true_label
NEGATIVE    200
POSITIVE    200
Name: count, dtype: int64


In [42]:
# load sentiment analysis models
distilbert_model = pipeline(
    "sentiment-analysis",
    model="distilbert-base-uncased-finetuned-sst-2-english"
)

roberta_model = pipeline(
    "sentiment-analysis",
    model="cardiffnlp/twitter-roberta-base-sentiment"
)

bert_model = pipeline(
    "sentiment-analysis",
    model="nlptown/bert-base-multilingual-uncased-sentiment"
)

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: cardiffnlp/twitter-roberta-base-sentiment
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

In [43]:
# convert model outputs into unified labels
def unify_label(label):

    label = str(label).upper()

    # distilbert labels
    if "POSITIVE" in label:
        return "POSITIVE"

    elif "NEGATIVE" in label:
        return "NEGATIVE"

    # roberta labels
    elif label == "LABEL_2":
        return "POSITIVE"

    elif label == "LABEL_0" or label == "LABEL_1":
        return "NEGATIVE"

    # bert labels
    elif "4" in label or "5" in label:
        return "POSITIVE"

    elif "1" in label or "2" in label or "3" in label:
        return "NEGATIVE"

    # default case
    else:
        return "NEGATIVE"

In [44]:
# keep final columns only
dataset = dataset[["text", "true_label"]]

# remove very short texts
dataset = dataset[dataset["text"].str.len() > 20]

# create prediction lists
distilbert_predictions = []
roberta_predictions = []
bert_predictions = []

distilbert_scores = []
roberta_scores = []
bert_scores = []

# run all texts on all models
for text in dataset["text"]:

    distilbert_result = distilbert_model(text)[0]
    distilbert_predictions.append(unify_label(distilbert_result["label"]))
    distilbert_scores.append(distilbert_result["score"])

    roberta_result = roberta_model(text)[0]
    roberta_predictions.append(unify_label(roberta_result["label"]))
    roberta_scores.append(roberta_result["score"])

    bert_result = bert_model(text)[0]
    bert_predictions.append(unify_label(bert_result["label"]))
    bert_scores.append(bert_result["score"])

In [45]:
# save all predictions in the dataset
dataset["DistilBERT_pred"] = distilbert_predictions
dataset["DistilBERT_score"] = distilbert_scores

dataset["RoBERTa_pred"] = roberta_predictions
dataset["RoBERTa_score"] = roberta_scores

dataset["BERT_pred"] = bert_predictions
dataset["BERT_score"] = bert_scores

# save predictions file
dataset.to_csv("predictions.csv", index=False)

# display predictions
print(dataset.head())

# define true labels
true_labels = dataset["true_label"]

# evaluate distilbert model
distilbert_accuracy = accuracy_score(true_labels, distilbert_predictions)
distilbert_precision = precision_score(true_labels, distilbert_predictions, pos_label="POSITIVE")
distilbert_recall = recall_score(true_labels, distilbert_predictions, pos_label="POSITIVE")
distilbert_f1 = f1_score(true_labels, distilbert_predictions, pos_label="POSITIVE")

# display results
print("\nDistilBERT Results:")
print(f"Accuracy: {distilbert_accuracy:.2f}")
print(f"Precision: {distilbert_precision:.2f}")
print(f"Recall: {distilbert_recall:.2f}")
print(f"F1 Score: {distilbert_f1:.2f}")

                                                text true_label  \
0  It's death-by-fear (aka scared-to-death) in th...   NEGATIVE   
1  As an Indie Horror flick director gathers his ...   NEGATIVE   
2  A singing notebook tells 3 puppets to be creat...   POSITIVE   
3  There are thin places between this world and t...   NEGATIVE   
4  In a post-apocalyptic world, a family is force...   POSITIVE   

  DistilBERT_pred  DistilBERT_score RoBERTa_pred  RoBERTa_score BERT_pred  \
0        POSITIVE          0.995902     NEGATIVE       0.646936  NEGATIVE   
1        NEGATIVE          0.998633     NEGATIVE       0.821005  NEGATIVE   
2        NEGATIVE          0.987092     NEGATIVE       0.525953  NEGATIVE   
3        POSITIVE          0.934616     NEGATIVE       0.537454  POSITIVE   
4        POSITIVE          0.843026     NEGATIVE       0.860774  POSITIVE   

   BERT_score  
0    0.383661  
1    0.327576  
2    0.415082  
3    0.302303  
4    0.426702  

DistilBERT Results:
Accuracy: 0.54
Pr

In [46]:
# evaluate roberta model
roberta_accuracy = accuracy_score(true_labels, roberta_predictions)
roberta_precision = precision_score(true_labels, roberta_predictions, pos_label="POSITIVE")
roberta_recall = recall_score(true_labels, roberta_predictions, pos_label="POSITIVE")
roberta_f1 = f1_score(true_labels, roberta_predictions, pos_label="POSITIVE")

# display results
print("RoBERTa Results:")
print(f"Accuracy: {roberta_accuracy:.2f}")
print(f"Precision: {roberta_precision:.2f}")
print(f"Recall: {roberta_recall:.2f}")
print(f"F1 Score: {roberta_f1:.2f}")

RoBERTa Results:
Accuracy: 0.51
Precision: 0.69
Recall: 0.04
F1 Score: 0.08


In [47]:
# evaluate bert model
bert_accuracy = accuracy_score(true_labels, bert_predictions)
bert_precision = precision_score(true_labels, bert_predictions, pos_label="POSITIVE")
bert_recall = recall_score(true_labels, bert_predictions, pos_label="POSITIVE")
bert_f1 = f1_score(true_labels, bert_predictions, pos_label="POSITIVE")

# display results
print("BERT Results:")
print(f"Accuracy: {bert_accuracy:.2f}")
print(f"Precision: {bert_precision:.2f}")
print(f"Recall: {bert_recall:.2f}")
print(f"F1 Score: {bert_f1:.2f}")

BERT Results:
Accuracy: 0.52
Precision: 0.51
Recall: 0.77
F1 Score: 0.61


In [48]:
# create comparison table
results = pd.DataFrame({

    "Model": ["DistilBERT", "RoBERTa", "BERT"],

    "Accuracy": [
        distilbert_accuracy,
        roberta_accuracy,
        bert_accuracy
    ],

    "Precision": [
        distilbert_precision,
        roberta_precision,
        bert_precision
    ],

    "Recall": [
        distilbert_recall,
        roberta_recall,
        bert_recall
    ],

    "F1 Score": [
        distilbert_f1,
        roberta_f1,
        bert_f1
    ]

})

# display results
print(results)

# save results table
results.to_csv("model_results.csv", index=False)

        Model  Accuracy  Precision  Recall  F1 Score
0  DistilBERT    0.5400   0.547619   0.460  0.500000
1     RoBERTa    0.5125   0.692308   0.045  0.084507
2        BERT    0.5150   0.509934   0.770  0.613546


In [49]:
# find best model
best_model = results.loc[results["F1 Score"].idxmax()]

print("Best Model:")
print(best_model)

Best Model:
Model            BERT
Accuracy        0.515
Precision    0.509934
Recall           0.77
F1 Score     0.613546
Name: 2, dtype: object


In [50]:
# test sentences for the demo
test_sentences = [

    "I absolutely loved this movie! The acting was brilliant and the story was amazing.",

    "This product is terrible. It broke after one day and the customer service was useless.",

    "The film had stunning visuals, but the plot made absolutely no sense whatsoever.",

    "I don't think this was a bad experience at all. I would definitely come back.",

    "It's not the worst thing I've ever tried, but I expected much more for the price."

]

In [51]:
# run test sentences on all models
for sentence in test_sentences:

    print("\nSentence:")
    print(sentence)

    print("\nDistilBERT:")
    print(distilbert_model(sentence)[0])

    print("\nRoBERTa:")
    print(roberta_model(sentence)[0])

    print("\nBERT:")
    print(bert_model(sentence)[0])

    print("\n----------------------------")


Sentence:
I absolutely loved this movie! The acting was brilliant and the story was amazing.

DistilBERT:
{'label': 'POSITIVE', 'score': 0.9998821020126343}

RoBERTa:
{'label': 'LABEL_2', 'score': 0.9922898411750793}

BERT:
{'label': '5 stars', 'score': 0.9484297633171082}

----------------------------

Sentence:
This product is terrible. It broke after one day and the customer service was useless.

DistilBERT:
{'label': 'NEGATIVE', 'score': 0.9998071789741516}

RoBERTa:
{'label': 'LABEL_0', 'score': 0.9834647178649902}

BERT:
{'label': '1 star', 'score': 0.9645270109176636}

----------------------------

Sentence:
The film had stunning visuals, but the plot made absolutely no sense whatsoever.

DistilBERT:
{'label': 'NEGATIVE', 'score': 0.9988150596618652}

RoBERTa:
{'label': 'LABEL_0', 'score': 0.7619836330413818}

BERT:
{'label': '2 stars', 'score': 0.52164626121521}

----------------------------

Sentence:
I don't think this was a bad experience at all. I would definitely come bac